# Velocity-Based Obstacle Detection

The purpose of this calculation is to answer the question:

> **"Will Peach's current direction of travel intersect the asteroid's safety region?"**

Rather than simply asking whether the asteroid is close, this method determines whether the asteroid actually lies **on Peach's current trajectory**.

---

# Step 1: Compute Peach's Velocity

```python
peach_velocity = Vector2(peach["velocity"])
peach_speed = peach_velocity.length()
```

## Mathematics

Peach's velocity is

$$
\mathbf{v} =
\begin{bmatrix}
v_x \\
v_y
\end{bmatrix}
$$

The magnitude of the velocity is

$$
|\mathbf{v}| = \sqrt{v_x^2 + v_y^2}
$$

## Geometric Meaning

This tells us two things:

- The **direction** Peach is moving.
- The **speed** Peach is moving.

```text
           ↑ vy

           |
           |
Peach •---->
        vx
```

---

# Step 2: Normalize the Velocity

```python
if peach_speed > 0:
    velocity_direction = peach_velocity.normalize()
else:
    velocity_direction = target_direction
```

## Mathematics

Normalization removes the magnitude:

$$
\hat{\mathbf v}
=
\frac{\mathbf v}{|\mathbf v|}
$$

After normalization,

$$
|\hat{\mathbf v}| = 1
$$

## Geometric Meaning

Instead of saying

> "Peach is moving 2.4 pixels/frame"

we now say

> "Peach is moving **in this direction**."

```text
Peach •────────────►
        unit vector
```

This unit vector defines Peach's current path.

---

# Step 3: Project the Asteroid onto Peach's Path

```python
asteroid_along_velocity = to_asteroid.dot(
    velocity_direction
)
```

## Mathematics

The dot product is

$$
\mathbf a \cdot \hat{\mathbf v}
=
|\mathbf a|
\cos\theta
$$

where

- $\mathbf a$ = vector from Peach to asteroid
- $\theta$ = angle between Peach's velocity and the asteroid.

Because

$$
|\hat{\mathbf v}| = 1,
$$

the result simplifies to

$$
\mathbf a \cdot \hat{\mathbf v}
=
|\mathbf a|\cos\theta
$$

which is exactly the **projection** of the asteroid onto Peach's direction of travel.

## Geometric Meaning

Imagine shining a flashlight straight ahead.

The asteroid casts a shadow onto Peach's path.

```text
                   Asteroid
                      •

Peach •----------------X------------->
```

The distance

```text
Peach → X
```

is

```python
asteroid_along_velocity
```

### Interpretation

If

$$
\texttt{asteroid\_along\_velocity} > 0
$$

the asteroid lies somewhere ahead.

If

$$
\texttt{asteroid\_along\_velocity} < 0
$$

the asteroid lies behind Peach.

---

# Step 4: Find the Closest Point on Peach's Path

```python
closest_point = (
    peach_position
    + velocity_direction * asteroid_along_velocity
)
```

## Mathematics

A point along a line can be written as

$$
P
=
P_0
+
d\hat{\mathbf v}
$$

where

- $P_0$ = Peach's position
- $d$ = projected distance
- $\hat{\mathbf v}$ = unit velocity vector

## Geometric Meaning

This reconstructs the point on Peach's future path that is closest to the asteroid.

```text
                 Asteroid
                     •

Peach •-------------X-------------->
```

The point **X** is the closest point on Peach's motion path.

---

# Step 5: Compute the Cross-Track Distance

```python
cross_track_distance = (
    asteroid_position - closest_point
).length()
```

## Mathematics

This computes

$$
|\mathbf r|
=
\sqrt{x^2+y^2}
$$

between

- the asteroid
- the closest point on the path.

## Geometric Meaning

This is the shortest distance from the asteroid to Peach's current trajectory.

```text
                 Asteroid
                     •

                     |
                     |
                     |

Peach •-------------X-------------->
```

This vertical distance is

```python
cross_track_distance
```

Notice that this is **not** the distance to Peach.

It is the distance to Peach's **future path**.

---

# Step 6: Determine Whether the Asteroid Blocks the Path

```python
asteroid_on_velocity_path = (
    asteroid_along_velocity > 0
    and cross_track_distance < effective_radius
)
```

## Mathematics

Two conditions must both be true.

### 1. Is the asteroid in front?

$$
\texttt{asteroid\_along\_velocity} > 0
$$

### 2. Does Peach's path intersect the asteroid's safety circle?

$$
\texttt{cross\_track\_distance}
<
\texttt{effective\_radius}
$$

---

## Geometric Meaning

Imagine expanding the asteroid by Peach's radius plus a safety margin.

```text
          ***********
       ***           ***
      **      •        **
       ***           ***
          ***********
```

Now extend Peach's velocity vector.

### Collision

```text
***************
      •
***************

Peach ------------------>
```

The path intersects the safety region.

Result:

```text
True
```

### No Collision

```text
***************
      •
***************


Peach ------------------------------>
```

The path misses the safety region.

Result:

```text
False
```

---

# Summary Diagram

```text
                     Asteroid
                         •

                         |
                         | cross_track_distance
                         |

Peach •------------------X-------------------------->
         velocity direction

       asteroid_along_velocity
```

The vector from Peach to the asteroid is decomposed into two components.

### Parallel Component

`asteroid_along_velocity`

Measures **how far ahead** the asteroid is along Peach's direction of travel.

### Perpendicular Component

`cross_track_distance`

Measures **how far away** the asteroid is from Peach's current path.

Together these answer the question:

> **"If Peach continues moving in its current direction, will it pass through the asteroid's safety region?"**

This decomposition appears throughout robotics, autonomous navigation, missile guidance, and path planning because it separates an obstacle's position into:

- **Progress along the path**
- **Deviation from the path**

These quantities are often much more useful than simply knowing the obstacle's distance and bearing.